# 第11章 指示チューニング

## 11.3 指示チューニングしたモデルの評価

In [1]:
!pip install flexeval bitsandbytes

### 11.3.1 モデルの動作確認

In [5]:
from flexeval import HuggingFaceLM
model_name = "llm-book/Swallow-7b-hf-oasst1-21k-ja"
llm = HuggingFaceLM(model=model_name)

2026-06-01 12:29:02.679 | INFO     | flexeval.core.language_model.hf_lm:__init__:229 - amp_dtype: None
2026-06-01 12:29:02.680 | INFO     | flexeval.core.language_model.hf_lm:__init__:230 - random seed: 42


In [ ]:
input_messages = [{"role": "user", "content": "1+1はなんでしょうか？"}]
print(llm.generate_chat_response(input_messages))

### 11.3.2 指示追従性能の評価

In [7]:
# このセルを実行すると、無料版のColabなどの低メモリ環境下ではRAMが不足しクラッシュする可能性があります
# その場合はランタイムを再起動し、動作確認をスキップして続きのセルを実行してください
import gc
import torch

# GPUに載せたモデルをCPUに移し、GPUを解放する
# HuggingFaceLMは遅延ロードのため、動作確認をしていない場合はllm.modelがNoneになる
if llm.model is not None:
    llm.model.cpu()
del llm
gc.collect()
torch.cuda.empty_cache()

In [8]:
from google.colab import drive

# Googleドライブを"drive"ディレクトリ以下にマウント
drive.mount("drive")

Mounted at drive


In [9]:
# 無料版のT4 GPUなど、低メモリ環境での評価コマンド
# 量子化とバッチサイズを小さく設定
!flexeval_lm \
  --language_model HuggingFaceLM \
  --language_model.model "llm-book/Swallow-7b-hf-oasst1-21k-ja" \
  --language_model.model_kwargs.load_in_4bit true \
  --eval_setup "vicuna-ja" \
  --eval_setup.gen_kwargs '{do_sample: True, temperature: 0.7, top_p: 0.9, max_new_tokens: 1024}' \
  --eval_setup.batch_size 1 \
  --save_dir "./drive/MyDrive/llm-book/IT_eval/vicuna-ja" \
  --force true

2026-06-01 12:29:56.333 | INFO     | flexeval.utils.module_utils:__call__:83 - Resolved config name 'vicuna-ja' to path '/usr/local/lib/python3.12/dist-packages/flexeval/preset_configs/EvalSetup/ja_chat/vicuna-ja.jsonnet'
2026-06-01 12:29:57.965 | INFO     | flexeval.scripts.flexeval_lm:main:222 - Namespace(language_model=Namespace(class_path='flexeval.HuggingFaceLM', init_args=Namespace(model='llm-book/Swallow-7b-hf-oasst1-21k-ja', model_kwargs={'load_in_4bit': True}, tokenizer=None, tokenizer_kwargs=None, add_special_tokens=False, amp_dtype=None, random_seed=42, load_peft=False, custom_chat_template=None, chat_template_kwargs=None, system_message=None, default_gen_kwargs=None, string_processors=None, model_limit_tokens='default', tool_parser=None, tools=None)), eval_setup=Namespace(class_path='flexeval.ChatResponse', init_args=Namespace(eval_dataset=Namespace(class_path='flexeval.ChatbotBench', init_args=Namespace(path_or_name='vicuna-ja', ref_path_or_name='vicuna-ja-ref-gpt4', need_

In [11]:
import json
from pathlib import Path

save_dir = "./drive/MyDrive/llm-book/IT_eval/vicuna-ja"

with open(Path(save_dir) / "outputs.jsonl") as f:
    for line in f:
        item = json.loads(line)
        print("===== 入力 ====")
        print(item["extra_info"]["messages"][0]["content"])
        print("===== モデル出力 ====")
        print(item["lm_output"])
        break   # 全件確認する場合は消してください

===== 入力 ====
時間管理能力を向上させるにはどうしたらいいですか？
===== モデル出力 ====
時間管理のテクニックには、以下のようなものがあります：

1.明確な目標を設定する：達成したいことを明確に理解することで、目標をより効果的に達成することができます。

2.ToDoリストを作る：タスクを管理し、必要なことを忘れないようにする。

3.優先順位をつける：重要度や緊急度に基づいてタスクの優先順位をつける。

4.時間を区切る：1日または1週間の時間枠を設定することで、集中力を維持し、重要な仕事を完了することができる。

5.タイマーを使う：タイマーを使って一定の時間を区切ることで、集中力を維持し、重要な仕事を完了することができる。

6.ポモドーロ・テクニックを使う：25分の作業と5分の休憩を3回繰り返す。

7.ポータブル・タイマーを使う：タイマーを見えるところに置くことで、時間を意識し、仕事を完了することができる。

8.ToDoアプリを使う：ToDoリスト、タスク管理、時間管理ができるToDoアプリを使う。

9.生産性向上ツールを使う：生産性向上ツールを使ってタスクを整理し、集中力を維持する。

10.習慣を作る：毎日同じ時間に同じことをする習慣を作り、時間を区切って集中し、重要な仕事を完了する。


In [12]:
!flexeval_presets assistant_eval_ja_single_turn

/*
This is a configuration for evaluting the quality of responses generated by an AI assistant.
Originally used to generate scores for the Japanese versions of MT-bench or Vicuna-bench.

Translated and adapted from [lm-sys/FastChat](https://github.com/lm-sys/FastChat/blob/main/fastchat/llm_judge/data/judge_prompts.jsonl).
*/
{
  class_path: 'ChatLLMScore',
  init_args: {
    language_model: { class_path: 'OpenAIChatAPI', init_args: { model: 'gpt-4-turbo-2024-04-09' } },
    valid_score_range: [1, 10],
    prompt_template: {
      class_path: 'Jinja2PromptTemplate',
      init_args: {
        template: std.stripChars(|||
          [指示]
          {% if references|length > 0 -%}
          以下に表示されるユーザの質問に対するアシスタントの応答の品質を評価してください。評価は正確さと有用性を考慮すべきです。アシスタントの回答の言語は、ユーザが使用している言語と一致しているべきで、そうでない場合は減点されるべきです。参照回答とアシスタントの回答が与えられます。あなたの評価は、アシスタントの回答と参照回答を比較することから始めてください。ミスを特定し、訂正してください。できるだけ客観的であること。評価の説明をした後、"[[rating]]"という形式で、1から10までの整数の評価値を出力してください（例 "rating：[[5]]"）。
          {%- else -%}
     

In [16]:
import os
import nest_asyncio
from flexeval import instantiate_from_config
from google.colab import userdata

# 先に環境変数へ API キーをセットする
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# nest_asyncio は引数なしで適用する
nest_asyncio.apply()

# この時点で OPENAI_API_KEY が解決できるためインスタンス化が成功する
metric = instantiate_from_config(
    "assistant_eval_ja_single_turn"
)

# 簡単な応答を評価
prompt = "元気よく挨拶してください"
lm_output = "こんにちは！！！！！！"
result = metric.evaluate(
    lm_outputs=[lm_output],
    extra_info_list=[
        {"messages": [{"role": "user", "content": prompt}]}
    ],
)

print("\n===== 評価者LLMの入力=====")
for message in result.instance_details[0]["llm_score_input"]:
    print(message["content"])
print("===== 評価者LLMの出力=====")
print(result.instance_details[0]["llm_score_output"])

2026-06-01 13:26:29.548 | INFO     | flexeval.utils.module_utils:__call__:83 - Resolved config name 'assistant_eval_ja_single_turn' to path '/usr/local/lib/python3.12/dist-packages/flexeval/preset_configs/Metric/assistant_eval_ja_single_turn.jsonnet'
Calculating ChatLLM score: 100%|██████████| 1/1 [00:08<00:00,  8.50s/it]


===== 評価者LLMの入力=====
あなたは優秀な助手です。
[指示]
以下に表示されるユーザの質問に対するアシスタントの応答の品質を公平に評価してください。評価は、応答の有用性、関連性、正確性、深さ、創造性、詳細度などの要素を考慮すべきです。アシスタントの回答の言語は、ユーザが使用している言語と一致しているべきで、そうでない場合は減点されるべきです。評価は短い説明から始めてください。できるだけ客観的であること。評価の説明をした後、"[[rating]]"という形式で、1から10までの整数の評価値を出力してください（例 "rating：[[5]]"）。

[ユーザの質問]
元気よく挨拶してください

[アシスタントの回答開始]
こんにちは！！！！！！
[アシスタントの回答終了]
===== 評価者LLMの出力=====
アシスタントの応答は要求された挨拶を行っていますが、多用された感嘆符がその文脈としてはやや過度かもしれません。しかし、全体としてユーザーの要求には応えていますが、創造性や詳細度の観点から見ると単純な応答であります。応答の言語はユーザの質問に合わせて適切です。

[[rating]]: [[6]]


In [19]:
!flexeval_file \
   --eval_file "./drive/MyDrive/llm-book/IT_eval/vicuna-ja/outputs.jsonl" \
   --metrics "assistant_eval_ja_single_turn" \
   --save_dir "./drive/MyDrive/llm-book/IT_eval/vicuna-ja/judge"

2026-06-01 13:27:42.179 | INFO     | flexeval.utils.module_utils:__call__:83 - Resolved config name 'assistant_eval_ja_single_turn' to path '/usr/local/lib/python3.12/dist-packages/flexeval/preset_configs/Metric/assistant_eval_ja_single_turn.jsonnet'
2026-06-01 13:27:43.422 | INFO     | flexeval.scripts.flexeval_file:main:137 - Namespace(eval_file='./drive/MyDrive/llm-book/IT_eval/vicuna-ja/outputs.jsonl', eval_data_loader=None, metrics=Namespace(class_path='flexeval.ChatLLMScore', init_args=Namespace(language_model=Namespace(class_path='flexeval.OpenAIChatAPI', init_args=Namespace(model='gpt-4-turbo-2024-04-09', api_headers=None, default_gen_kwargs=None, developer_message=None, string_processors=None, model_limit_new_tokens=None, max_parallel_requests=None, tools=None, max_num_trials=None, first_wait_time=None, max_wait_time=None, backend='OpenAI')), prompt_template=Namespace(class_path='flexeval.Jinja2PromptTemplate', init_args=Namespace(template='[指示]\n{% if references|length > 0 -%

In [20]:
!cat drive/MyDrive/llm-book/IT_eval/vicuna-ja/judge/metrics.json

{
    "llm_score": 5.1875,
    "num_failed_score_parses": 0,
    "elapsed_time": 509.3436714649997
}

In [23]:
!cat drive/MyDrive/llm-book/IT_eval/vicuna-ja/judge/outputs.jsonl

{"lm_output": "時間管理のテクニックには、以下のようなものがあります：\n\n1.明確な目標を設定する：達成したいことを明確に理解することで、目標をより効果的に達成することができます。\n\n2.ToDoリストを作る：タスクを管理し、必要なことを忘れないようにする。\n\n3.優先順位をつける：重要度や緊急度に基づいてタスクの優先順位をつける。\n\n4.時間を区切る：1日または1週間の時間枠を設定することで、集中力を維持し、重要な仕事を完了することができる。\n\n5.タイマーを使う：タイマーを使って一定の時間を区切ることで、集中力を維持し、重要な仕事を完了することができる。\n\n6.ポモドーロ・テクニックを使う：25分の作業と5分の休憩を3回繰り返す。\n\n7.ポータブル・タイマーを使う：タイマーを見えるところに置くことで、時間を意識し、仕事を完了することができる。\n\n8.ToDoアプリを使う：ToDoリスト、タスク管理、時間管理ができるToDoアプリを使う。\n\n9.生産性向上ツールを使う：生産性向上ツールを使ってタスクを整理し、集中力を維持する。\n\n10.習慣を作る：毎日同じ時間に同じことをする習慣を作り、時間を区切って集中し、重要な仕事を完了する。", "finish_reason": "stop", "extra_info": {"category": "generic", "messages": [{"role": "user", "content": "時間管理能力を向上させるにはどうしたらいいですか？"}]}, "references": [], "output_length": 516, "llm_score": 7, "llm_score_input": [{"role": "system", "content": "あなたは優秀な助手です。"}, {"role": "user", "content": "[指示]\n以下に表示されるユーザの質問に対するアシスタントの応答の品質を公平に評価してください。評価は、応答の有用性、関連性、正確性、深さ、創造性、詳細度などの要素を考慮すべきです。アシスタントの回答の言語は、ユーザが使用している言語と一致しているべきで、そうでない場合は減点されるべきです。評価は短い説明から始め

In [24]:
import json
from pathlib import Path

save_dir = "./drive/MyDrive/llm-book/IT_eval/vicuna-ja/judge"

with open(Path(save_dir) / "outputs.jsonl") as f:
    eval_items = [json.loads(line) for line in f]

def print_eval_item(item):
    print("===== 評価者LLMの入力=====")
    print(item["llm_score_input"][1]["content"])
    print("===== 評価者LLMの出力=====")
    print(item["llm_score_output"])

for item in eval_items:
    print_eval_item(item)
    print("\n\n")
    break

===== 評価者LLMの入力=====
[指示]
以下に表示されるユーザの質問に対するアシスタントの応答の品質を公平に評価してください。評価は、応答の有用性、関連性、正確性、深さ、創造性、詳細度などの要素を考慮すべきです。アシスタントの回答の言語は、ユーザが使用している言語と一致しているべきで、そうでない場合は減点されるべきです。評価は短い説明から始めてください。できるだけ客観的であること。評価の説明をした後、"[[rating]]"という形式で、1から10までの整数の評価値を出力してください（例 "rating：[[5]]"）。

[ユーザの質問]
時間管理能力を向上させるにはどうしたらいいですか？

[アシスタントの回答開始]
時間管理のテクニックには、以下のようなものがあります：

1.明確な目標を設定する：達成したいことを明確に理解することで、目標をより効果的に達成することができます。

2.ToDoリストを作る：タスクを管理し、必要なことを忘れないようにする。

3.優先順位をつける：重要度や緊急度に基づいてタスクの優先順位をつける。

4.時間を区切る：1日または1週間の時間枠を設定することで、集中力を維持し、重要な仕事を完了することができる。

5.タイマーを使う：タイマーを使って一定の時間を区切ることで、集中力を維持し、重要な仕事を完了することができる。

6.ポモドーロ・テクニックを使う：25分の作業と5分の休憩を3回繰り返す。

7.ポータブル・タイマーを使う：タイマーを見えるところに置くことで、時間を意識し、仕事を完了することができる。

8.ToDoアプリを使う：ToDoリスト、タスク管理、時間管理ができるToDoアプリを使う。

9.生産性向上ツールを使う：生産性向上ツールを使ってタスクを整理し、集中力を維持する。

10.習慣を作る：毎日同じ時間に同じことをする習慣を作り、時間を区切って集中し、重要な仕事を完了する。
[アシスタントの回答終了]
===== 評価者LLMの出力=====
この回答では時間管理に関する一連の具体的な方法が提供されています。提案された方法はよく知られており、実用的であるため、ユーザーが時間管理能力を向上させるために直ぐに実行することができます。特に、ポモドーロテクニックやタイマーの使用、タスクの優先順位つけな

In [26]:
print(eval_items[0].keys())

dict_keys(['lm_output', 'finish_reason', 'extra_info', 'references', 'output_length', 'llm_score', 'llm_score_input', 'llm_score_output'])


In [34]:
for i in range(1, 30, 3):
    print(eval_items[i]["extra_info"]["category"])

generic
generic
generic
knowledge
knowledge
knowledge
knowledge
roleplay
roleplay
roleplay


In [30]:
from collections import defaultdict
from pprint import pprint

# カテゴリごとのスコアを集計
scores_per_category = defaultdict(list)
for item in eval_items:
    category = item["extra_info"]["category"]
    scores_per_category[category].append(item["llm_score"])
# カテゴリごとの平均スコアを計算
avg_scores = {
    cat: sum(scores) / len(scores)
    for cat, scores in scores_per_category.items()
}
# スコアの高い順に表示
for category, avg_score in sorted(
    avg_scores.items(), key=lambda x: x[1], reverse=True
):
    print(category, round(avg_score, 1))

knowledge 6.7
common-sense 6.7
generic 6.6
writing 6.2
roleplay 5.8
counterfactual 4.8
fermi 3.2
coding 1.6
math 1.3


In [36]:
eval_items.sort(key=lambda x: x["llm_score"], reverse=True)
print_eval_item(eval_items[0])

===== 評価者LLMの入力=====
[指示]
以下に表示されるユーザの質問に対するアシスタントの応答の品質を公平に評価してください。評価は、応答の有用性、関連性、正確性、深さ、創造性、詳細度などの要素を考慮すべきです。アシスタントの回答の言語は、ユーザが使用している言語と一致しているべきで、そうでない場合は減点されるべきです。評価は短い説明から始めてください。できるだけ客観的であること。評価の説明をした後、"[[rating]]"という形式で、1から10までの整数の評価値を出力してください（例 "rating：[[5]]"）。

[ユーザの質問]
多文化社会において、言語や文化の壁は、人々のコミュニケーションや人間関係の形成にどのような影響を与えますか？

[アシスタントの回答開始]
言語や文化の壁は、多文化社会において、人々のコミュニケーションや人間関係の形成に大きな影響を与えます。言語や文化の壁は、人々が互いを理解し、効果的にコミュニケーションをとる能力に影響を与え、人間関係の形成を難しくする可能性があります。

多文化社会における言語や文化の壁が、人々のコミュニケーションや人間関係の形成に与える影響には、以下のようなものがある：

1.情報伝達の困難：異なる言語を話す人々は、互いに効果的にコミュニケーションをとることが難しく、情報の伝達が困難になることがある。

2.誤解や偏見の可能性：異なる言語や文化を持つ人々は、誤解や偏見を生み、効果的なコミュニケーションを妨げる可能性がある。

3.社会的孤立：異なる言語や文化を持つ人々は、社会的なつながりや人間関係を形成することが難しく、社会的孤立につながることがある。

4.異文化への適応の難しさ：異なる文化的背景を持つ人々は、新しい環境に適応することが難しく、孤立感や疎外感を感じることがある。

5.相互尊重の欠如：異なる言語や文化を持つ人々は、互いの違いを尊重することが難しく、偏見や差別を生み出す可能性がある。

多文化社会における言語や文化の壁を克服し、より包括的で協力的な社会を構築するためには、以下のような対策を講じることができる：

1.言語教育と文化理解の促進：言語教育と文化理解の促進：異なる言語や文化を持つ人々が互いを理解し、効果的にコミュニケーションをとることを助けるために、言語教育と文化理解を

In [37]:
print_eval_item(eval_items[-1])

===== 評価者LLMの入力=====
[指示]
以下に表示されるユーザの質問に対するアシスタントの応答の品質を公平に評価してください。評価は、応答の有用性、関連性、正確性、深さ、創造性、詳細度などの要素を考慮すべきです。アシスタントの回答の言語は、ユーザが使用している言語と一致しているべきで、そうでない場合は減点されるべきです。評価は短い説明から始めてください。できるだけ客観的であること。評価の説明をした後、"[[rating]]"という形式で、1から10までの整数の評価値を出力してください（例 "rating：[[5]]"）。

[ユーザの質問]
適切なフォーマットで、コンピュータサイエンスの名門大学院に出願する学生への正式な推薦状を書いてください。

[アシスタントの回答開始]
適切なフォーマットで、コンピュータサイエンスの名門大学院に出願する学生への正式な推薦状を書いてください。
[アシスタントの回答終了]
===== 評価者LLMの出力=====
ユーザーがアシスタントに求めたのは、コンピュータサイエンスの名門大学院への出願用の推薦状のフォーマット提供でした。しかし、アシスタントは単にユーザーの質問を繰り返すだけの応答をしました。この応答はユーザーの要求に全く応じておらず、有用性、関連性、正確性、深さ、創造性、詳細度のいずれの観点からも評価が非常に低いです。

評価: [[1]]
